In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GraphSAGE
from sklearn.linear_model import LogisticRegression
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [12]:
# Load the datasets
cora_dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = cora_dataset[0]
data = data.to(device, 'x', 'edge_index')

train_loader = LinkNeighborLoader(
    data,
    batch_size=256,
    shuffle=True,
    neg_sampling_ratio=1.0,
    num_neighbors=[10, 10],
)

model = GraphSAGE(
    data.num_node_features,
    hidden_channels=64,
    num_layers=2,
).to(device)

### somehow i have to get all of this data to deeprobust for it to be used in the attack
model.nclass=data.y.argmax() + 1
model.nfeat=data.num_node_features
model.hidden_sizes=[64]
model.with_relu=True
model.output=None
model.best_model = None
model.best_output = None

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [13]:
def train():
    model.train()

    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        h = model(batch.x, batch.edge_index)
        h_src = h[batch.edge_label_index[0]]
        h_dst = h[batch.edge_label_index[1]]
        pred = (h_src * h_dst).sum(dim=-1)
        loss = F.binary_cross_entropy_with_logits(pred, batch.edge_label)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * pred.size(0)

    return total_loss / data.num_nodes

In [14]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index).cpu()

    clf = LogisticRegression()
    clf.fit(out[data.train_mask], data.y[data.train_mask])

    val_acc = clf.score(out[data.val_mask], data.y[data.val_mask])
    test_acc = clf.score(out[data.test_mask], data.y[data.test_mask])

    return val_acc, test_acc

In [15]:
for epoch in range(0, 200):
    loss = train()
    acc = test()[1]
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 4.5307, Accuracy: 0.6180
Epoch: 001, Loss: 3.8450, Accuracy: 0.6280
Epoch: 002, Loss: 3.7322, Accuracy: 0.6480
Epoch: 003, Loss: 3.6849, Accuracy: 0.6280
Epoch: 004, Loss: 3.6457, Accuracy: 0.6180
Epoch: 005, Loss: 3.6586, Accuracy: 0.6180
Epoch: 006, Loss: 3.6446, Accuracy: 0.6140
Epoch: 007, Loss: 3.6658, Accuracy: 0.6200
Epoch: 008, Loss: 3.6165, Accuracy: 0.6090
Epoch: 009, Loss: 3.5892, Accuracy: 0.5990
Epoch: 010, Loss: 3.5724, Accuracy: 0.6070
Epoch: 011, Loss: 3.6101, Accuracy: 0.5900
Epoch: 012, Loss: 3.5770, Accuracy: 0.5820
Epoch: 013, Loss: 3.5621, Accuracy: 0.5830
Epoch: 014, Loss: 3.6073, Accuracy: 0.5670
Epoch: 015, Loss: 3.5928, Accuracy: 0.5940
Epoch: 016, Loss: 3.5880, Accuracy: 0.5810
Epoch: 017, Loss: 3.5635, Accuracy: 0.5720
Epoch: 018, Loss: 3.5391, Accuracy: 0.5680
Epoch: 019, Loss: 3.5032, Accuracy: 0.5760
Epoch: 020, Loss: 3.5873, Accuracy: 0.5720
Epoch: 021, Loss: 3.5469, Accuracy: 0.5600
Epoch: 022, Loss: 3.5739, Accuracy: 0.5810
Epoch: 023,

In [50]:
torch.save(model.state_dict(), 'cora_gsage.pt')